In [9]:
import sys
from typing import Optional
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

# 1. Import your tools
from tools.csv_tool import read_csv_data

# 2. Define the exact Output Schemas for your 3 Database Tables
class SoilRecord(BaseModel):
    year: int = Field(description="Observation year")
    crop: str = Field(description="Crop name. If missing, use 'UNSPECIFIED'")
    b_level: float = Field(alias="B", description="Boron level")
    category: str = Field(description="Soil category")
    cu_level: float = Field(alias="Cu", description="Copper level")
    district: str = Field(description="District name")
    district_id: str = Field(description="District ID code")
    ec_level: float = Field(alias="EC", description="Electrical Conductivity")
    k_level: float = Field(alias="K", description="Potassium level")
    mn_level: float = Field(alias="Mn", description="Manganese level")
    n_level: float = Field(alias="N", description="Nitrogen level")
    organic_carbon: float = Field(description="Organic Carbon percentage")
    phosphorous: float = Field(description="Phosphorous level")
    tehsil: str = Field(description="Tehsil name")
    village: str = Field(description="Village name")
    zn_level: float = Field(alias="Zn", description="Zinc level")
    fe_level: float = Field(alias="Fe", description="Iron level")
    ph_level: float = Field(alias="pH", description="pH level")

class RainfallRecord(BaseModel):
    district_id: str = Field(description="District ID code")
    year_month: str = Field(description="Format YYYY-MM")
    year: int = Field(description="Year of observation")
    month: str = Field(description="Month of observation")
    rainfall_mm: float = Field(description="Rainfall in mm")
    rainy_days_over_2_5mm: int = Field(description="Number of rainy days > 2.5mm")

class ProductionRecord(BaseModel):
    year: int = Field(description="Year of production")
    area: float = Field(description="Area of cultivation")
    production: float = Field(description="Total production volume")
    yield_value: float = Field(description="Yield amount")
    crop: str = Field(description="Crop name, normalized")
    district: str = Field(description="District name")

# 3. The Master Schema
class MasterExtraction(BaseModel):
    soil_data: Optional[list[SoilRecord]] = Field(default=None, description="Fill this ONLY if the source contains soil health metrics.")
    rainfall_data: Optional[list[RainfallRecord]] = Field(default=None, description="Fill this ONLY if the source contains rainfall/weather data.")
    production_data: Optional[list[ProductionRecord]] = Field(default=None, description="Fill this ONLY if the source contains crop production/yield data.")

def main():
    if len(sys.argv) < 3 or sys.argv[1] != "input":
        print("Usage: python run.py input <file_path>")
        sys.exit(1)
        
    input_source = sys.argv[2]
    
    llm = ChatOpenAI(model="gpt-4o", temperature=0)
    tools = [read_csv_data] 
    
    # 4. The Engineered System Prompt
    system_prompt = """You are an elite, autonomous agricultural data extraction pipeline agent operating in a strictly deterministic manner. Your primary objective is to ingest raw source files, categorize them, and extract metrics with zero data loss or hallucination.

    OPERATING PROTOCOL:
    1. TOOL EXECUTION: Look at the user's input string. You must use the appropriate available tool (e.g., `read_csv_data`) to read the file contents.
    
    2. CLASSIFICATION: Analyze the raw data headers. Categorize the file into exactly one of three domains:
       - SOIL HEALTH (Detects: pH, Boron, EC, Organic Carbon, Nitrogen, etc.)
       - RAINFALL (Detects: Rainfall(mm), Rainy_Days, Month, etc.)
       - PRODUCTION (Detects: Area, Production, Yield, Crop, etc.)
    
    3. DATA NORMALIZATION & EXTRACTION:
       - Extract all valid rows. Ignore summary footers or irrelevant metadata headers.
       - Standardize all crop names to standard English conventions (e.g., "Paddy" becomes "Rice").
       - MISSING VALUE RULE: If any text-based field (like Crop, District, or Village) is missing or blank in the source data, you MUST substitute the exact string 'UNSPECIFIED'. Do not leave it blank.
       - Do not invent numerical values. If a number is entirely missing, omit the row if it breaks the schema, or substitute 0.0 if logically appropriate for the metric.
       
    4. OUTPUT PREPARATION: Return a comprehensive textual summary of the extracted records. Ensure your output clearly separates the rows so the downstream JSON parser can map them flawlessly to the target schema.
    """
    
    agent_executor = create_react_agent(llm, tools, state_modifier=system_prompt)
    
    print(f"Agent is analyzing {input_source}...")
    agent_response = agent_executor.invoke({
        "messages": [HumanMessage(content=f"Extract data from this source: {input_source}")]
    })
    
    raw_agent_text = agent_response["messages"][-1].content
    
    print("\nStructuring output into the correct database schema...")
    structured_llm = llm.with_structured_output(MasterExtraction)
    final_structured_data = structured_llm.invoke(
        f"Format this raw extraction. Map it to the correct list based on your classification. Data: {raw_agent_text}"
    )
    
    print("\n--- FINAL DB-READY OUTPUT ---")
    print(final_structured_data.model_dump_json(indent=2, exclude_none=True))

if __name__ == "__main__":
    main()

Usage: python run.py input <file_path>


SystemExit: 1

C:\Users\Romir Karna\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [5]:
%pip install tools

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
